#PT-P2-Neural Network Training Exercise

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


import pandas as pd
import numpy as np
import tensorflow as tf

from tensorflow.keras.callbacks import EarlyStopping #Early Stopping feature

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GlobalAveragePooling1D, Dense, Dropout
from tensorflow.keras.optimizers import Adam, SGD

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score

from collections import Counter

import matplotlib.pyplot as plt

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

This code cell **prepares the main tools and settings needed for the neural network experiments**. It begins by connecting Google Colab to my Google Drive through `drive.mount('/content/drive')`, which allows the notebook to access the dataset files stored there.

The next group of imports provides the libraries used throughout the exercise. `pandas` is used for loading and handling the CSV dataset, `numpy` supports numerical arrays and data processing, while `tensorflow` provides the framework for building and training the neural network. From Keras, `Sequential` is used to create the model layer by layer, while `Embedding`, `GlobalAveragePooling1D`, `Dense`, and `Dropout` define the architecture. The `Adam` and `SGD` optimizers are included so different optimization settings can be tested during the experiments.

I also imported `EarlyStopping`, which became useful during the custom experiments because it allows training to stop when `val_loss` no longer improves. From Scikit-learn, `train_test_split()` is used to divide the dataset into training, validation, and testing sets. The functions `f1_score()`, `accuracy_score()`, `precision_score()`, and `recall_score()` provide the evaluation metrics used to measure the model's performance. `Counter` is later used when building the vocabulary, while `matplotlib.pyplot` is used to create the training and validation loss curves.

Finally, `np.random.seed(42)` and `tf.random.set_seed(42)` set the random seed for NumPy and TensorFlow. I kept the value at `42` throughout the experiments to make the training process more reproducible and to reduce unnecessary random differences when comparing different model configurations. Overall, this cell prepares the libraries, evaluation tools, and reproducibility settings needed for the rest of the notebook.

## 1. LOAD & PREPROCESS CSV DATA

In [ ]:
#CSV_FILEPATH = "/content/drive/MyDrive/PT-P2-Neural Network Training_TULIO/10examples_TULIO.csv"
CSV_FILEPATH = "/content/drive/MyDrive/PT-P2-Neural Network Training_TULIO/Custom_Expanded_Training_Dataset_1.csv"

MAX_LEN = 20              # Fixed sequence length (padded/truncated)
MAX_VOCAB_SIZE = 1000     # Vocabulary size limit

This code cell sets the **dataset location and the main text preprocessing limits** that will be used before training. `CSV_FILEPATH` contains the location of the CSV file in Google Drive. The original `10examples_TULIO.csv` path is kept as a comment for reference, while `Custom_Expanded_Training_Dataset_1.csv` is the dataset currently selected for the custom experiments.

The `MAX_LEN` value is set to `20`, which means each feedback entry will be represented using a maximum of 20 tokens. Shorter sequences will later be padded, while longer sequences will be shortened to this limit. `MAX_VOCAB_SIZE` is set to `1000` to control the maximum number of words that can be included in the vocabulary.

Keeping these values as separate variables also makes experimentation easier. For example, I can adjust `MAX_LEN` or `MAX_VOCAB_SIZE` when testing different text preprocessing configurations without changing the functions that use them.

In [ ]:
# Load CSV
df = pd.read_csv(CSV_FILEPATH)

This code cell **loads the selected dataset into the notebook**. The `pd.read_csv()` function reads the CSV file located at `CSV_FILEPATH` and stores its contents in the DataFrame `df`.

After loading the file, `df` becomes the main reference for accessing the dataset. In the succeeding cells, I use columns such as `df["text"]` for the customer feedback and `df["label"]` for their corresponding categories. This step is simple, but it is important because the preprocessing process depends on the dataset being loaded correctly first.

In [ ]:
# Basic Tokenizer + Vocabulary Builder
def build_vocab(texts, max_vocab_size):
    words = [word for text in texts for word in str(text).split()]
    word_counts = Counter(words)

    # Reserve index 0 for padding (<PAD>) and 1 for unknown tokens (<UNK>)
    most_common = word_counts.most_common(max_vocab_size - 2)

    vocab = {
        word: idx + 2
        for idx, (word, _) in enumerate(most_common)
    }

    vocab["<PAD>"] = 0
    vocab["<UNK>"] = 1

    return vocab

This code cell defines the `build_vocab()` function, which is responsible for **creating the vocabulary used to represent words as numerical values**. It starts by going through all of the provided `texts`, splitting each text into individual words, and combining those words into one collection.

The `Counter()` function then counts how often each word appears. From these counts, `most_common()` keeps the most frequently occurring words according to `max_vocab_size`. I noticed that the code subtracts `2` from the vocabulary limit because two positions are intentionally reserved for special values.

The vocabulary assigns normal words numerical IDs starting from `2`. The value `0` is reserved for `<PAD>`, which will be used when a sequence is shorter than the required length, while `1` is assigned to `<UNK>` for words that are not available in the vocabulary.

Once these values are assigned, the function returns `vocab`. This gives the next preprocessing step a consistent way to convert words from the feedback into numerical IDs that the neural network can work with.

In [ ]:
def text_to_sequence(text, vocab, max_len):
    tokens = str(text).lower().split()

    seq = [vocab.get(token, 1) for token in tokens]

    if len(seq) < max_len:
        seq += [0] * (max_len - len(seq))
    else:
        seq = seq[:max_len]

    return seq

This code cell defines the `text_to_sequence()` function, which **converts each feedback entry from text into a fixed-length numerical sequence**. The text is first converted to lowercase using `.lower()` and then separated into individual tokens using `.split()`. Converting the text to lowercase helps keep words such as "Good" and "good" from being treated differently during this step.

Each token is then matched with its numerical ID from `vocab`. The expression `vocab.get(token, 1)` looks for the token in the vocabulary, and if it cannot be found, the value `1` is used to represent `<UNK>` or an unknown word.

The next part makes sure that every sequence has the same length. When the sequence is shorter than `max_len`, zeros are added as `<PAD>` values. If the sequence is longer, `seq[:max_len]` keeps only the allowed number of tokens.

The completed sequence is then returned. This is necessary because the neural network expects the input samples to have a consistent numerical structure rather than different text lengths.

In [ ]:
# Build vocab and encode text
vocab = build_vocab(df["text"].tolist(), MAX_VOCAB_SIZE)

encoded_texts = [
    text_to_sequence(txt, vocab, MAX_LEN)
    for txt in df["text"]
]

This code cell **applies the vocabulary and sequence conversion functions to the actual feedback dataset**. First, `build_vocab()` receives the values from `df["text"]` and the `MAX_VOCAB_SIZE` limit. The `.tolist()` method converts the text column into a list that the vocabulary-building function can process.

After the vocabulary is created, the list comprehension goes through every feedback entry in `df["text"]`. Each entry is passed to `text_to_sequence()` together with `vocab` and `MAX_LEN`.

The result is stored in `encoded_texts`. At this point, the original feedback sentences have been transformed into fixed-length sequences of numerical IDs. This is where the text starts becoming suitable as input for the neural network.

In [ ]:
X_data = np.array(encoded_texts)
y_data = np.array(df["label"].values)

num_classes = len(np.unique(y_data))

This code cell **prepares the input features and target labels in numerical array form**. `np.array(encoded_texts)` converts all of the processed text sequences into the NumPy array `X_data`, which will serve as the input data for the neural network.

For the expected outputs, `df["label"].values` retrieves the labels from the dataset and converts them into the NumPy array `y_data`. This separates the inputs and their corresponding correct classifications into two variables that can later be passed to the training process.

The last part uses `np.unique(y_data)` to identify the different class labels present in the dataset. Applying `len()` gives the total number of classes and stores it in `num_classes`. Since my dataset contains three categories, this value is later used to create the correct number of neurons in the model's output layer.

In [ ]:
# Dataset Partitioning (70% Train, 15% Val, 15% Test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X_data,
    y_data,
    test_size=0.30,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42
)

This code cell **divides the processed dataset into training, validation, and testing sets**. The first `train_test_split()` separates `X_data` and `y_data`, keeping 70% of the samples for `X_train` and `y_train`. The remaining 30% is temporarily stored in `X_temp` and `y_temp`.

A second `train_test_split()` divides that remaining 30% equally. Half becomes `X_val` and `y_val` for validation, while the other half becomes `X_test` and `y_test` for testing. This results in the intended distribution of **70% training, 15% validation, and 15% testing**.

I kept `random_state=42` in both splits so that the data is divided consistently when the code is rerun. The training set is used to teach the model, the validation set allows me to monitor its performance while experimenting with different configurations, and the test set remains separate for evaluating how well the trained model handles unseen data.

## 2. HYPERPARAMETER CONFIGURATION

In [ ]:
CONFIG = {
    "run_id": "EXP-01-KERAS-BASELINE",
    "embed_dim": 32,
    "hidden_dim": 64,
    "learning_rate": 0.005,
    "batch_size": 32,
    "epochs": 15,
    "optimizer_type": "Adam",
    "dropout_rate": 0.1
}

This code cell **defines the hyperparameter configuration used for the current training experiment**. I placed the settings inside the `CONFIG` dictionary so they can be changed in one location without modifying the other parts of the neural network code.

The `run_id` identifies the experiment and makes it easier to track which configuration produced a particular result. `embed_dim` controls the size of the numerical representation learned for each word, while `hidden_dim` determines the number of neurons in the hidden Dense layer.

The training behavior is controlled by several other values. `learning_rate` determines how large the weight adjustments are during training, `batch_size` specifies how many training samples are processed before the model updates its weights, and `epochs` sets the maximum number of times the model can go through the complete training set. The `optimizer_type` selects the optimizer used for updating the weights.

Finally, `dropout_rate` controls how much Dropout regularization is applied to the hidden layer. Keeping these settings together became especially useful during experimentation because I could adjust specific hyperparameters and observe how each change affected the training and validation performance.

## 3. KERAS SEQUENTIAL ARCHITECTURE

In [ ]:
model = Sequential()

This code cell **creates the neural network model using Keras `Sequential()`**. At this point, the model does not contain any layers yet. It serves as the structure where the Embedding, pooling, hidden, regularization, and output layers will be added in order.

I used a Sequential model because the architecture follows a straightforward flow where the output of one layer is passed directly to the next layer. This makes the structure easier to build and observe during the experiments.

In [ ]:
# Embedding Layer (Pillar 1: Architecture)
model.add(
    Embedding(
        input_dim=len(vocab),
        output_dim=CONFIG["embed_dim"],
        input_length=MAX_LEN,
        mask_zero=True
    )
)

This code cell adds the **Embedding layer**, which is the first layer responsible for processing the numerical text sequences. The `input_dim` is based on `len(vocab)`, so the layer knows how many vocabulary entries can appear in the input.

The `output_dim` uses `CONFIG["embed_dim"]`, which controls the size of the numerical vector learned for each word. Instead of treating a word as only an integer ID, the Embedding layer learns a set of numerical features that can represent useful information about that word during training.

`input_length=MAX_LEN` connects the layer to the fixed sequence length prepared during preprocessing. I also use `mask_zero=True` because the value `0` is reserved for `<PAD>`. This tells the model that padded positions are not actual words.

I consider this part important to the Architecture pillar because changing `embed_dim` directly changes how much information the model can learn to represent for each vocabulary word.

In [ ]:
# Global Average Pooling across token sequence dimension
model.add(GlobalAveragePooling1D())

This code cell adds `GlobalAveragePooling1D()`, which **combines the sequence of word embeddings into one fixed-size representation**. The Embedding layer produces a numerical vector for each token in the feedback, so there are several vectors for a single input sequence.

`GlobalAveragePooling1D()` takes the average of these token representations across the sequence. This gives the next Dense layer one compact representation of the feedback instead of a separate vector for every token position.

From observing the architecture, this also keeps the model relatively simple because it reduces the sequence before it reaches the hidden layer. The resulting features can then be used by the Dense layer for classification.

In [ ]:
# First Hidden Layer
model.add(Dense(CONFIG["hidden_dim"], activation="relu"))

This code cell adds the **main hidden Dense layer** of the neural network. The number of neurons comes from `CONFIG["hidden_dim"]`, which means its size can be changed easily between experiments.

The layer uses the `relu` activation function. ReLU allows the network to learn more complex patterns from the numerical text representation rather than relying only on simple linear relationships.

The `hidden_dim` value became one of the important architecture settings during experimentation. Increasing or decreasing the number of neurons changes the capacity of the model, so I could observe whether a larger hidden layer helped the model learn the feedback patterns more effectively.

In [ ]:
# Regularization Layer (Pillar 3: Regularization)
model.add(Dropout(CONFIG["dropout_rate"]))

This code cell adds the **Dropout regularization layer** after the hidden Dense layer. The amount of Dropout is controlled by `CONFIG["dropout_rate"]`.

During training, Dropout temporarily disables a portion of the hidden-layer outputs. This encourages the network to avoid depending too heavily on only a few neurons and can help reduce overfitting.

I found this setting useful during the experiments because a model can continue improving on the training data while becoming less effective on validation data. Adjusting the dropout rate provides a way to control that behavior and observe whether stronger regularization improves generalization.

In [ ]:
# Output Layer
model.add(Dense(num_classes, activation="softmax"))

This code cell adds the **final output layer used for classification**. The number of neurons is determined by `num_classes`, which was calculated earlier from the unique labels in the dataset. Since the feedback task contains three categories, the model produces one output value for each possible class.

The layer uses the `softmax` activation function, which converts the outputs into class probabilities. These probabilities show how strongly the model associates the input feedback with each category.

During prediction, the class with the highest probability can then be selected as the model's final classification. This completes the forward structure of the neural network, starting from the encoded text and ending with the predicted feedback category.

## 4. OPTIMIZER & COMPILATION

In [ ]:
if CONFIG["optimizer_type"] == "Adam":
    opt = Adam(learning_rate=CONFIG["learning_rate"])
else:
    opt = SGD(learning_rate=CONFIG["learning_rate"])

This code cell **selects the optimizer that will be used to update the neural network's weights during training**. It checks the value stored in `CONFIG["optimizer_type"]`. If the selected optimizer is `"Adam"`, the code creates an `Adam` optimizer. Otherwise, it uses `SGD`.

Both choices use `CONFIG["learning_rate"]`, so the learning rate can also be adjusted from the configuration. The learning rate controls the size of the weight adjustments made as the model learns from its errors.

Keeping this selection connected to `CONFIG` makes the optimization process easier to experiment with. I can change the optimizer or learning rate without rewriting the training code, which helps keep the different experiment configurations organized.

In [ ]:
# Configure loss function, optimizer engine, and evaluation metrics
model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=opt,
    metrics=["accuracy"]
)

This code cell **compiles the neural network and defines how its training performance will be measured**. The loss function is set to `sparse_categorical_crossentropy`, which is appropriate for this task because there are multiple possible classes and the target labels are represented as integer values.

The `optimizer=opt` setting uses the optimizer selected in the previous code cell. During training, this optimizer adjusts the model's weights based on the calculated loss and the configured learning rate.

I also include `metrics=["accuracy"]` so the model reports the proportion of correctly classified samples during both training and validation. At this stage, the architecture and learning settings are connected together, so the model is ready to begin training.

In [ ]:
model.summary()

This code cell uses `model.summary()` to **display the structure of the neural network before training begins**. The summary shows the layers included in the model together with their output shapes and parameter counts once the model has been built.

I use this output as a quick check that the architecture matches the current experiment. This becomes useful when values such as `embed_dim` or `hidden_dim` are changed because the summary provides a visible confirmation of how those architecture settings affect the model.

## 5. THE OPERATION LOOP (FIT & EVALUATE)

In [ ]:
print(f"--- Starting Training Run: {CONFIG['run_id']} ---")

This code cell **displays the ID of the experiment before the training process begins**. It retrieves the value from `CONFIG["run_id"]` and includes it in the printed message.

Since I performed several training experiments with different configurations, showing the run ID makes it easier to identify which experiment produced the following epoch results. It also helps keep the outputs organized when comparing the different training runs.

In [ ]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

This code cell configures **Early Stopping**, which I added to prevent the model from continuing to train when its validation performance is no longer improving. The callback monitors `val_loss`, so it specifically watches the validation loss after each epoch.

The `patience=3` setting allows the model to continue for three consecutive epochs without an improvement in validation loss. This gives the model some opportunity to improve again instead of stopping immediately after one weaker epoch.

I also use `restore_best_weights=True`. This became an important part of the experiments because the epoch where training stops is not always the epoch with the best validation performance. With this option enabled, Keras restores the weights from the epoch that achieved the lowest validation loss.

Finally, `verbose=1` makes the Early Stopping activity visible in the output. This allowed me to observe both the stopping epoch and the earlier best epoch that was restored.

In [ ]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=CONFIG["epochs"],
    batch_size=CONFIG["batch_size"],
    callbacks=[early_stopping],
    verbose=1
)

This code cell **starts the actual neural network training process using `model.fit()`**. The model learns from `X_train` and `y_train`, which contain the training inputs and their correct labels.

The `validation_data=(X_val, y_val)` argument allows the model to evaluate itself on the validation set after every epoch. I use this to compare training performance with data that is not directly being used to update the model's weights.

The maximum number of training epochs comes from `CONFIG["epochs"]`, while `CONFIG["batch_size"]` determines how many training samples are processed before a weight update is performed. The `callbacks=[early_stopping]` setting activates the Early Stopping configuration from the previous cell.

The results from every completed epoch are stored in `history`. I use this object later to retrieve the training and validation losses for the loss-curve graph. One thing I observed during the custom experiments is that the actual number of completed epochs can be lower than the configured maximum because Early Stopping can end the training once validation loss stops improving.

In [ ]:
# Compute final validation predictions for F1-score calculation
val_preds_probs = model.predict(X_val)
val_preds = np.argmax(val_preds_probs, axis=1)

This code cell **generates predictions for the validation dataset after training has finished**. `model.predict(X_val)` processes each validation sample and returns the probability assigned to every possible class. These probability values are stored in `val_preds_probs`.

Since I need actual class predictions for the evaluation metrics, `np.argmax()` is used with `axis=1` to find the class with the highest probability for each sample. The selected class values are stored in `val_preds`.

Because these predictions are generated after `model.fit()` finishes, they use the model's current weights. When Early Stopping restores the best weights, the validation predictions are therefore based on that restored model rather than simply the final stopping epoch.

In [ ]:
val_f1 = f1_score(
    y_val,
    val_preds,
    average="weighted"
)

This code cell calculates the **weighted F1-score for the validation predictions**. The `f1_score()` function compares the actual validation labels in `y_val` with the model's predictions in `val_preds`.

The F1-score combines precision and recall into one measurement, which gives me another way to evaluate the model beyond accuracy alone. I use `average="weighted"` so the contribution of each class is considered according to the number of samples belonging to that class.

I included this metric in the experiment results because it provides a more balanced view of classification performance, especially when comparing different model configurations.

In [ ]:
# Validation Weighted Precision
val_precision = precision_score(
    y_val,
    val_preds,
    average="weighted",
    zero_division=0
)

This code cell calculates the **weighted precision of the model on the validation set**. `precision_score()` compares `y_val` with `val_preds` and measures how often the model's predicted class assignments are correct.

The `average="weighted"` setting combines the precision values from all classes while considering how many validation samples belong to each class. I also use `zero_division=0` so the calculation returns `0` instead of producing an undefined result if the model does not predict a particular class.

Adding precision gives me a clearer view of the quality of the model's predictions and provides another metric that can be recorded in the training log.

In [ ]:
# Validation Weighted Recall
val_recall = recall_score(
    y_val,
    val_preds,
    average="weighted",
    zero_division=0
)

This code cell calculates the **weighted recall for the validation set**. Recall focuses on how many of the actual samples belonging to each class were successfully identified by the model.

The function compares `y_val` with `val_preds`, while `average="weighted"` combines the recall values across all classes according to their number of samples. Similar to the precision calculation, `zero_division=0` prevents an undefined value from interrupting the evaluation if a class has no valid calculation.

Together with precision and F1-score, recall gives me a more complete view of how well the model performs across the feedback categories instead of relying only on overall accuracy.

In [ ]:
val_loss = history.history["val_loss"][-1]

This code cell retrieves the **last recorded validation loss** from the training history. `history.history["val_loss"]` contains the validation-loss value from every completed epoch, while `[-1]` selects the final value in that list.

While working with Early Stopping, I observed that this value does not always represent the best epoch. For example, the model may achieve its lowest validation loss at an earlier epoch and then stop several epochs later because of the `patience=3` setting.

Since `restore_best_weights=True` restores the earlier best model, I also check the Early Stopping output when recording the best validation result. This helped me distinguish between the **best epoch** and the **early stopping epoch** during the custom experiments.

In [ ]:
# Final Log Output summary for Excel logging
print(
    f"ID: {CONFIG['run_id']} | "
    f"LR: {CONFIG['learning_rate']} | "
    f"Hidden: {CONFIG['hidden_dim']} | "
    f"Dropout: {CONFIG['dropout_rate']} | "
    f"Final Val Loss: {val_loss:.4f} | "
    f"Final Val F1: {val_f1:.4f} | "
    f"Final Val Precision: {val_precision:.4f} | "
    f"Final Val Recall: {val_recall:.4f} | "
)

This code cell **prints a summary of the main configuration and validation results for the current experiment**. It displays `run_id`, learning rate, hidden-layer size, and dropout rate so I can immediately see which settings were used.

The output also includes the validation loss, weighted F1-score, weighted precision, and weighted recall. The `:.4f` formatting keeps these numerical results to four decimal places, which makes them easier to read and record consistently.

I added this summary because the exercise requires the results from each experiment to be documented in the training log. Having the important values together in one output makes it easier to transfer the results and compare how the different configurations affected model performance.

## 6. GENERATE LOSS CURVES CHART

In [ ]:
train_losses = history.history["loss"]
val_losses = history.history["val_loss"]

This code cell **retrieves the training and validation loss values recorded during each completed epoch**. The values come from the `history` object that was returned by `model.fit()`.

`history.history["loss"]` contains the training loss and is stored in `train_losses`, while `history.history["val_loss"]` contains the validation loss and is stored in `val_losses`.

Keeping both sets of values allows me to compare how the model behaves on the training data and validation data over time. This comparison is useful for observing whether the model is learning properly or beginning to show signs of underfitting or overfitting.

In [ ]:
#epochs_range = range(1, CONFIG["epochs"] + 1)
epochs_range = range(1, len(train_losses) + 1)

This code cell creates the **epoch range that will be used along the X-axis of the loss-curve graph**. Originally, the range was based on `CONFIG["epochs"]`, which assumes that the model always completes every configured epoch.

After adding Early Stopping, I changed the range to `len(train_losses)`. This checks how many epochs were actually completed and creates the X-axis based on that number.

This adjustment became necessary during the custom experiments. For example, a model configured for 50 epochs might stop at Epoch 11. Using the actual number of recorded training-loss values prevents the graph from expecting 50 points when only 11 epochs were completed.

In [ ]:
plt.figure(figsize=(8, 5))

This code cell **creates the figure that will contain the loss curves**. The `figsize=(8, 5)` setting controls the width and height of the graph.

I used this size to keep the graph large enough for the epoch values, loss curves, labels, and legend to remain readable without taking up too much space in the notebook.

In [ ]:
plt.plot(
    epochs_range,
    train_losses,
    label="Training Loss",
    color="blue",
    linewidth=2,
    marker="o"
)

This code cell **plots the training loss across the completed epochs**. `epochs_range` provides the epoch numbers along the X-axis, while `train_losses` provides the corresponding training-loss values along the Y-axis.

The curve is labeled `"Training Loss"` so it can be identified in the graph legend. I also use a solid line with circular markers to make the individual epoch values easier to observe.

Watching this curve helped me see how quickly the model was learning the training data. In several experiments, the training loss became very small, so comparing it with validation loss was important for determining whether that improvement also carried over to unseen validation samples.

In [ ]:
plt.plot(
    epochs_range,
    val_losses,
    label="Validation Loss",
    color="red",
    linewidth=2,
    linestyle="--",
    marker="s"
)

This code cell adds the **validation-loss curve** to the same graph. It uses the same `epochs_range`, but the Y-axis values come from `val_losses`.

I gave the validation curve a different line style and square markers so it can be distinguished clearly from the training-loss curve. This curve became especially important when using Early Stopping because `val_loss` is the value being monitored to determine whether training should continue.

By comparing both curves, I can observe whether validation loss continues to decrease with training loss or begins to rise while training loss keeps falling. The second pattern can indicate that the model is beginning to overfit the training data.

In [ ]:
plt.title(
    f"Performance Monitoring: Loss Curves ({CONFIG['run_id']})",
    fontsize=12,
    fontweight="bold"
)

This code cell **adds a title to the loss-curve graph**. The title includes `CONFIG["run_id"]`, so the graph automatically displays the ID of the experiment that produced it.

I found this helpful when comparing several experiments because each graph can be identified without having to rely only on its position in the notebook. The `fontsize` and `fontweight` settings also make the title easier to notice above the plotted results.

In [ ]:
plt.xlabel("Epochs", fontsize=10)
plt.ylabel("Loss (Sparse Categorical Cross-Entropy)", fontsize=10)

This code cell **labels the X-axis and Y-axis of the loss graph**. `plt.xlabel()` labels the horizontal axis as `"Epochs"`, which represents how many complete training cycles have been performed.

`plt.ylabel()` identifies the vertical values as the loss measured using **Sparse Categorical Cross-Entropy**, which is the same loss function configured earlier in `model.compile()`.

These labels make the graph easier to interpret because they clearly show that I am observing how the model's calculated loss changes as training progresses through each epoch.

In [ ]:
plt.xticks(epochs_range)

This code cell **displays each completed epoch as a value on the X-axis**. The values come directly from `epochs_range`, so the displayed ticks match the actual number of epochs completed during training.

This is particularly useful with Early Stopping because I can clearly identify where the model stopped and compare that point with the best epoch reported in the training output.

In [ ]:
plt.grid(True, linestyle=":", alpha=0.6)
plt.legend(loc="upper right")
plt.tight_layout()

This code cell **improves the readability and layout of the completed graph**. `plt.grid()` adds light reference lines, which make it easier to compare the loss values across different epochs.

`plt.legend()` displays the labels for `"Training Loss"` and `"Validation Loss"` in the upper-right portion of the graph. This makes it clear which curve represents each type of loss.

Finally, `plt.tight_layout()` automatically adjusts the spacing around the graph. I use this so the title, axis labels, tick values, and legend fit properly without overlapping or being cut off.

In [ ]:
plt.show()

This code cell **displays the completed training and validation loss graph**. Once `plt.show()` is called, I can visually compare how both losses changed throughout the training process.

I use this graph together with the numerical metrics in the training log when evaluating each experiment. A large or increasing separation between training and validation loss can suggest overfitting, while consistently high losses can indicate underfitting. When both curves decrease and validation performance remains strong, the model shows a better fit.

For the custom experiments, the graph also helped me understand why Early Stopping selected a particular best epoch. It provides a visual confirmation of where validation loss stopped improving even when training loss continued to decrease.

#List of changes to each EXP for this exercise

In [ ]:
# ============================================================
# EXPERIMENT CHANGE LOG
# ============================================================

# EXP-01 – KERAS BASELINE
# Purpose: Establish the initial baseline performance of the model.
# Dataset: Original dataset (30 samples / 10 samples per class)
# embed_dim = 32
# hidden_dim = 64
# learning_rate = 0.005
# batch_size = 32
# epochs = 15
# optimizer = Adam
# dropout_rate = 0.1
# MAX_LEN = 20
# MAX_VOCAB_SIZE = 1000
# No custom optimization was applied.


# EXP-02 – LOWER MODEL CAPACITY / LEARNING RATE
# Purpose: Test the effect of reducing model capacity and using a
# smaller learning rate.
# Changes from EXP-01:
# hidden_dim: 64 -> 16
# learning_rate: 0.005 -> 0.0001
# dropout_rate: 0.1 -> 0.0
# Other settings remained unchanged.


# EXP-03 – HIGH MODEL CAPACITY / HIGH LEARNING RATE
# Purpose: Test the effect of a much larger hidden layer and
# aggressive learning rate.
# Changes:
# hidden_dim: 16 -> 256
# learning_rate: 0.0001 -> 0.05
# dropout_rate remained 0.0
# Result showed severe overfitting / unstable generalization.


# EXP-04 – REGULARIZATION
# Purpose: Reduce overfitting by introducing stronger dropout.
# Changes:
# hidden_dim: 256 -> 128
# learning_rate: 0.05 -> 0.005
# dropout_rate: 0.0 -> 0.4
# Dropout was used to limit dependence on specific neurons.


# ============================================================
# CUSTOM OPTIMIZATION EXPERIMENTS
# ============================================================

# EXP-05A – DATA EXPANSION + EARLY STOPPING
# Strategy: Data Augmentation / Expansion
# Main change:
# Dataset: 30 samples -> 900 balanced samples
#          (300 samples per class)
#
# Configuration:
# embed_dim = 32
# hidden_dim = 128
# learning_rate = 0.005
# batch_size = 32
# epochs = 30 maximum
# dropout_rate = 0.4
# MAX_LEN = 20
# MAX_VOCAB_SIZE = 1000
#
# EarlyStopping was enabled:
# monitor = "val_loss"
# patience = 3
# restore_best_weights = True
#
# Best Epoch = 3
# Early Stopping Epoch = 6


# EXP-05B – OPTIMIZATION & TRAINING
# Strategy: Lower learning rate with longer maximum training.
# Main changes from EXP-05A:
# learning_rate: 0.005 -> 0.001
# epochs: 30 -> 50 maximum
#
# Dataset remained at 900 samples.
# hidden_dim = 128
# dropout_rate = 0.4
# batch_size = 32
# EarlyStopping remained enabled.
#
# Best Epoch = 19
# Early Stopping Epoch = 22


# EXP-05C – ARCHITECTURE OPTIMIZATION
# Strategy: Increase hidden layer capacity.
# Main change from EXP-05B:
# hidden_dim: 128 -> 256
#
# learning_rate = 0.001
# embed_dim = 32
# batch_size = 32
# dropout_rate = 0.4
# epochs = 50 maximum
# MAX_LEN = 20
# MAX_VOCAB_SIZE = 1000
# EarlyStopping remained enabled.
#
# Best Epoch = 14
# Early Stopping Epoch = 17


# EXP-05D – TEXT PREPROCESSING
# Strategy: Adjust maximum sequence length.
# Main change from EXP-05C:
# MAX_LEN: 20 -> 30
#
# MAX_VOCAB_SIZE remained 1000.
# embed_dim = 32
# hidden_dim = 256
# learning_rate = 0.001
# batch_size = 32
# dropout_rate = 0.4
# EarlyStopping remained enabled.
#
# Best Epoch = 14
# Early Stopping Epoch = 17


# EXP-05E – ARCHITECTURE + OPTIMIZATION
# Strategy: Combine architecture and optimization changes.
#
# Main changes:
# MAX_LEN: 30 -> 20
# embed_dim: 32 -> 64
# batch_size: 32 -> 16
#
# hidden_dim = 256
# learning_rate = 0.001
# dropout_rate = 0.4
# MAX_VOCAB_SIZE = 1000
# epochs = 50 maximum
# EarlyStopping remained enabled.
#
# Best Epoch = 8
# Early Stopping Epoch = 11


# EXP-05F – ARCHITECTURE + TEXT PREPROCESSING
# Strategy: Combine a larger embedding representation with
# a larger vocabulary limit.
#
# Main changes from EXP-05E:
# embed_dim: 64 -> 128
# MAX_VOCAB_SIZE: 1000 -> 2000
#
# MAX_LEN = 20
# hidden_dim = 256
# learning_rate = 0.001
# batch_size = 16
# dropout_rate = 0.4
# epochs = 50 maximum
# EarlyStopping remained enabled.
#
# Best Epoch = 8
# Early Stopping Epoch = 11